<a href="https://colab.research.google.com/github/bonsai/ashizawa_stories/blob/main/wagahai_novel_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI=CAT: Wagahai Novel Generator Pipeline
このノートブックは、元のBashスクリプトで記述された小説生成パイプラインをGoogle Colab環境向けに最適化したものです。
各セルを順番に実行していくことで、データの品質評価からファインチューニングまでを一気通貫で行うことができます。

## パイプライン全体の定数定義と初期設定
まずは処理全体で使用するファイルパスの定義と、環境のセットアップ（Phase 0）を行います。

In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import os
import shutil

repo_dir = 'ashizawa_stories'

print(f'Checking for existing repository: {repo_dir}')
if os.path.exists(repo_dir):
    print(f'Removing existing directory: {repo_dir}')
    shutil.rmtree(repo_dir)

print('Cloning the repository...')
!git clone https://github.com/bonsai/ashizawa_stories.git
print('Repository cloned successfully.')

# List contents of the data directory to confirm path
print('\nContents of ashizawa_stories/data/twnovel_roseaullus:')
!ls -R ashizawa_stories/data/twnovel_roseaullus/


Checking for existing repository: ashizawa_stories
Cloning the repository...
Cloning into 'ashizawa_stories'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 79 (delta 25), reused 55 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 743.29 KiB | 14.29 MiB/s, done.
Resolving deltas: 100% (25/25), done.
Repository cloned successfully.

Contents of ashizawa_stories/data/twnovel_roseaullus:
ashizawa_stories/data/twnovel_roseaullus/:
twnovel_dataset_roseaullus.csv	twnovel_work_roseaullus.txt


In [17]:
import os
import sys

# パイプライン全体の定数定義
INPUT_CSV = "/content/ashizawa_stories/data/twnovel_roseaullus/twnovel_dataset_roseaullus.csv"
CLEANED_CSV = "cleaned_dataset.csv"
NOVELTY_CSV = "analysis_result_with_novelty.csv"
MERGED_DATA = "./wagahai_theme/merged_wagahai_dataset.csv"

print("=========================================")
print("  AI=CAT: Wagahai Novel Generator Pipeline")
print("=========================================")

print("[Phase 0] 環境セットアップ...")
# 00_setup_colab.py を実行
!python ashizawa_stories/00_setup_colab.py

  AI=CAT: Wagahai Novel Generator Pipeline
[Phase 0] 環境セットアップ...
Colab/Kaggle 環境セットアップを開始します...
=== Google Drive の空き容量確認 ===
Filesystem      Size  Used Avail Use% Mounted on
drive            15G   12G  3.8G  76% /content/drive

利用可能な容量: 3.8G
Drive の容量確認完了

=== 依存ライブラリのインストール ===
✓ pandas は既にインストールされています
✓ numpy は既にインストールされています
✓ transformers は既にインストールされています
✓ torch は既にインストールされています
✓ accelerate は既にインストールされています
✓ datasets は既にインストールされています

=== ワークスペースの設定 ===
プロジェクトディレクトリは既に存在します: /content/drive/MyDrive/twnovel_ml_project

=== セットアップ完了 ===
プロジェクトルート: /content/drive/MyDrive/twnovel_ml_project
データ保存先: /content/drive/MyDrive/twnovel_ml_project/data
モデル保存先: /content/drive/MyDrive/twnovel_ml_project/models

次のステップ: 01_evaluate_and_filter.py を実行してデータのクレンジングを行ってください。


## Phase 1: データの品質評価とフィルタリング
入力となるCSVファイルが存在するかチェックし、存在すれば評価およびクリーニングのスクリプトを実行します。

In [3]:
print("[Phase 1] データの品質評価とフィルタリング...")

# Execute the cleaning script by first changing the directory in the shell.
# This ensures the `python` script runs with `/content/ashizawa_stories/data/` as its CWD,
# allowing it to find `twnovel_roseaullus/twnovel_dataset_roseaullus.csv` correctly.
!cd /content/ashizawa_stories/ && python 01_evaluate_and_filter.py --input_file twnovel_roseaullus/twnovel_dataset_roseaullus.csv

[Phase 1] データの品質評価とフィルタリング...
小説データの品質評価とフィルタリングを開始します...

データを読み込んでいます: twnovel_roseaullus/twnovel_dataset_roseaullus.csv
読み込んだデータ数: 490

=== ルールベースフィルタリング ===

ルールベースフィルタリング完了:
  初期データ数：490
  除外数：0
  残存数：490

=== Perplexity 計算中 (モデル：rinna/japanese-gpt2-small) ===
使用するデバイス：cuda
モデルを読み込んでいます...
config.json: 100% 846/846 [00:00<00:00, 3.61MB/s]
tokenizer_config.json: 100% 282/282 [00:00<00:00, 1.65MB/s]
special_tokens_map.json: 100% 153/153 [00:00<00:00, 962kB/s]
spiece.model: 100% 806k/806k [00:00<00:00, 972kB/s] 
model.safetensors: 100% 454M/454M [00:04<00:00, 94.4MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 1217.93it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: rinna/japanese-gpt2-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can

## Phase 2: 新奇性分析とクラスタリング
前段階で作成されたクリーンデータを用いて、新奇性の分析とクラスタリングを実行します。

In [4]:
print("[Phase 2] 新奇性分析とクラスタリング...")

if not os.path.exists(CLEANED_CSV):
    print(f"エラー: クリーンデータ {CLEANED_CSV} が見つかりません。")
    sys.exit(1)

# 新奇性分析スクリプトの実行
!python ashizawa_stories/02_novelty_analysis.py --input_file {CLEANED_CSV}

[Phase 2] 新奇性分析とクラスタリング...


NameError: name 'CLEANED_CSV' is not defined

## Phase 3: 学習用データの整形と分割
分析結果のCSVを元に、機械学習モデルのトレーニングに適した形式への変換とデータの分割を行います。

In [ ]:
print("[Phase 3] 学習用データの整形と分割...")

if not os.path.exists(NOVELTY_CSV):
    print(f"エラー: 新奇性分析結果 {NOVELTY_CSV} が見つかりません。")
    sys.exit(1)

# トレーニングデータ準備スクリプトの実行
!python ashizawa_stories/03_prepare_training.py --input_file {NOVELTY_CSV}

## Phase 4: 「吾輩はAIである」テーマのデータ強化
特定のテーマ（吾輩はAIである）に沿ったデータの水増しや拡張、マージ処理を行います。

In [ ]:
print("[Phase 4] '吾輩はAIである' テーマのデータ強化...")

if not os.path.exists(NOVELTY_CSV):
    print(f"エラー: 新奇性分析結果 {NOVELTY_CSV} が見つかりません。")
    sys.exit(1)

# データ強化スクリプトの実行
!python ashizawa_stories/04_setup_wagahai.py --input_file {NOVELTY_CSV} --merge

## Phase 5: ファインチューニング（学習の実行）
Phase 4 でマージされた専用データが存在する場合はそれを使用し、存在しない場合はPhase 3の標準的なデータを使用してファインチューニングを実行します。

In [ ]:
print("[Phase 5] ファインチューニング開始...")

# 条件に応じたデータパスとエポック数の設定
if os.path.exists(MERGED_DATA):
    print("マージ済みデータを使用して学習します...")
    train_data_path = MERGED_DATA
else:
    print("マージ済みデータが見つからないため、Phase 3 のデータで学習します...")
    train_data_path = "./training_data/train.jsonl"

# トレーニングスクリプトの実行
!python ashizawa_stories/05_train_wagahai.py --data_dir {train_data_path} --output_dir "./wagahai_ai_model" --num_epochs 3

print("=========================================")
print("  パイプライン完了！")
print("  生成モデル: ./wagahai_ai_model/")
print("=========================================")